# T1-U7C: audit the completed experiment

The frozen intervention failed the all-development-seed gate (3/6 passed). This notebook verifies the portable results without training, tuning, probing, or opening reserved seeds 1103/1201/1301. Upload `T1_U7C_PORTABLE_BUNDLE.zip`. The audit takes several CPU minutes; no GPU is required.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, zipfile

uploaded = files.upload()
name = "T1_U7C_PORTABLE_BUNDLE.zip"
if name not in uploaded:
    raise ValueError(f"Upload {name}")
archive = Path(name).resolve()
print("Bundle SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())


In [ ]:
destination = Path(tempfile.mkdtemp(prefix="t1_u7c_audit_"))
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        if not (destination / member.filename).resolve().is_relative_to(destination):
            raise ValueError("Unsafe archive path")
    bundle.extractall(destination)
project = destination / "t1_transformer_trajectory"
manifest = project / "outputs/t1_u7c/T1_U7C_INNER_SHA256SUMS.txt"
for entry in manifest.read_text().splitlines():
    expected, relative = entry.split("  ", 1)
    target = (project / relative).resolve()
    if not target.is_relative_to(project):
        raise ValueError("Unsafe manifest path")
    assert hashlib.sha256(target.read_bytes()).hexdigest() == expected, relative
print("All", len(manifest.read_text().splitlines()), "manifest entries verified.")


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(project / "requirements.txt")])
print("Recorded environment:")
print((project / "outputs/t1_u7c/environment.json").read_text())


In [ ]:
subprocess.check_call([sys.executable, "-u", "audit_t1_u7c.py"], cwd=project)
audit = json.loads((project / "outputs/t1_u7c/recovery_audit.json").read_text())
print(audit["status"])
print(audit["decision"])


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
output = project / "outputs/t1_u7c"
display(pd.read_csv(output / "development_seed_gate.csv"))
display(Image(filename=str(output / "t1_u7c_training_signal_repair.png")))
display(Markdown((output / "T1_U7C_REPORT.md").read_text()))


Original outputs remain intact. The audit updates only its own audit record in this extracted temporary copy. Differences versus T1-U7B for seeds 811/907/1009 are paired by model seed, not by held-out example; see the report. Final checkpoints store weights only. No training replay is part of this notebook.
